# CLIFFGUARD — round 3

Use a T4 GPU, then Run all. Expected runtime is about 5–6 hours if no retries are needed. This notebook settles whether the HH-RLHF FP16-versus-4-bit comparison remains at a 256-token window, and supplies 256-token FP16 XSTest baselines for all three models.

It deliberately does not run 5.5 bits, because this round isolates the deployable 4-bit comparison. It does not run GSM8K, because the decisive behavioural and labelled-arm gaps are cheaper to settle first. It does not run AWQ or GPTQ, because their installation and checkpoint paths are fragile on an unattended free-tier session.

Caches and completed runs are checkpointed to Drive. Re-running after a disconnect restores completed work and does not create a second run with the same label.

## Environment

The setup is deliberately the same as round 2. It mounts Drive before any model work, so completed schemes survive a Colab disconnect.

In [ ]:
import os, sys, json, time, pathlib, platform, subprocess

IN_COLAB = 'google.colab' in sys.modules
REPO_URL = 'https://github.com/parnish007/CLIFFGUARD.git'
REPO_DIR = pathlib.Path('/content/CLIFFGUARD') if IN_COLAB else pathlib.Path.cwd()
DRIVE_ROOT = pathlib.Path('/content/drive/MyDrive/cliffguard')

if IN_COLAB:
    try:
        from google.colab import drive as _drive
        _drive.mount('/content/drive')
        DRIVE_ROOT.mkdir(parents=True, exist_ok=True)
    except Exception as exc:
        print('[drive] not mounted — a disconnect will lose progress:', exc)
    if not REPO_DIR.exists():
        subprocess.run(['git', 'clone', '--depth', '1', REPO_URL, str(REPO_DIR)], check=True)
    os.chdir(REPO_DIR)
    subprocess.run([sys.executable, '-m', 'pip', '-q', 'install', 'bitsandbytes', 'datasets', 'gguf'], check=False)

if str(REPO_DIR) not in sys.path:
    sys.path.insert(0, str(REPO_DIR))
import torch, numpy as np, transformers
HAS_GPU = torch.cuda.is_available()
GPU_NAME = torch.cuda.get_device_name(0) if HAS_GPU else 'NONE'
VRAM_GB = round(torch.cuda.get_device_properties(0).total_memory / 1e9, 2) if HAS_GPU else 0.0
print(f'repo         : {pathlib.Path.cwd()}')
print(f'python       : {platform.python_version()}')
print(f'torch        : {torch.__version__}')
print(f'transformers : {transformers.__version__}')
print(f'numpy        : {np.__version__}')
print(f'GPU          : {GPU_NAME}  ({VRAM_GB} GB)')
if hasattr(os, 'statvfs'):
    st = os.statvfs('.')
    print(f'free disk    : {st.f_bavail * st.f_frsize / 1e9:.1f} GB')
if not HAS_GPU:
    raise SystemExit('No GPU. Runtime → Change runtime type → T4 GPU, then rerun this cell.')
if tuple(int(p) for p in transformers.__version__.split('.')[:2]) < (4, 45):
    raise SystemExit(f'transformers {transformers.__version__} too old (need >= 4.45).\nRun: !pip -q install -U transformers, then restart the runtime.')


## Preflight

This gate runs before any expensive model load. Do not start the measurements if it reports a failure.

In [ ]:
# Equivalent to: !python scripts/preflight_round2.py
# subprocess makes a non-zero shell exit fatal; a notebook shell magic need not.
preflight = subprocess.run([sys.executable, 'scripts/preflight_round2.py'])
if preflight.returncode != 0:
    raise SystemExit('PREFLIGHT FAILED. Do not start round 3; repair the reported gate failure first.')


In [ ]:
MODELS_LONG = [('qwen3b', 'Qwen/Qwen2.5-3B-Instruct'), ('phi35', 'microsoft/Phi-3.5-mini-instruct')]
MODELS_XSTEST = MODELS_LONG + [('smollm17b', 'HuggingFaceTB/SmolLM2-1.7B-Instruct')]
JUDGE_MODEL = 'Qwen/Qwen2.5-7B-Instruct'
N_LONG, N_XSTEST, LONG_TOKENS, SEED = 250, 150, 256, 0
JUDGE_COMPLETION_CHARS, TAXONOMY_MAX_LENGTH = 2000, 2560

# Caches go straight to Drive when it is mounted. The old mirror-between-arms
# design left completed schemes on Colab's disposable disk until a whole model
# had finished, so a disconnect could lose hours of valid cache entries.
CACHE_ROOT = (DRIVE_ROOT / 'artifacts') if DRIVE_ROOT.exists() else pathlib.Path('artifacts')
BEHAV_CACHE = str(CACHE_ROOT / 'behavioural_cache')
RESULTS = {}
print(f'long HH-RLHF : {N_LONG} per class, {LONG_TOKENS} tokens, FP16 + RTN 4-bit')
print(f'XSTest       : {N_XSTEST} per class, {LONG_TOKENS} tokens, FP16 only')
print(f'caches       : {CACHE_ROOT}' + ('' if DRIVE_ROOT.exists() else '   (LOCAL -- a disconnect loses them)'))

def run_step(label, script, args, timeout=10800):
    '''Stream one script invocation; keep its tail and exit status.

    The timeout is a watchdog rather than proc.wait(timeout=...). Reading a
    child's stdout to EOF blocks for as long as it lives, so wait was reached
    only after exit and could never stop a hung step. A watchdog kill is kept
    separate because Linux reports it as -9, the same code as an OOM kill.
    '''
    import threading
    cmd = [sys.executable, f'scripts/{script}'] + args
    print(f'\n$ {" ".join(cmd)}', flush=True)
    started, lines = time.time(), []
    proc = subprocess.Popen(cmd, stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True, bufsize=1)
    timed_out = []
    def _kill():
        timed_out.append(True)
        print(f'\n[{label}] no exit after {timeout / 3600:.1f} h; killing', flush=True)
        proc.kill()
    watchdog = threading.Timer(timeout, _kill)
    watchdog.daemon = True
    watchdog.start()
    try:
        for line in proc.stdout:
            if 'Loading weights' in line or 'it/s]' in line or 's/prompt' in line:
                continue
            lines.append(line.rstrip())
            print(line.rstrip(), flush=True)
        proc.wait()
    except Exception as exc:
        proc.kill()
        proc.wait()
        lines.append(f'ABORTED: {type(exc).__name__}: {exc}')
    finally:
        watchdog.cancel()
    ok = proc.returncode == 0
    RESULTS[label] = {'returncode': proc.returncode, 'timed_out': bool(timed_out), 'minutes': (time.time() - started) / 60, 'tail': lines[-40:]}
    print(f'\n=== {label}: {"OK" if ok else f"FAILED rc={proc.returncode}"} in {RESULTS[label]["minutes"]:.1f} min ===', flush=True)
    return ok

def run_step_resumable(label, script, args, attempts=3, timeout=10800):
    '''Retry only a real OOM. Each completed scheme is cached immediately, so a
    fresh process resumes farther through the ladder instead of starting over.
    A bad flag, missing checkpoint, or timeout cannot be improved by retrying.'''
    tag = label
    for attempt in range(1, attempts + 1):
        tag = label if attempt == 1 else f'{label}-retry{attempt}'
        if attempt > 1:
            print(f'\n[retry {attempt}/{attempts}] {label}: resuming from cache', flush=True)
        if run_step(tag, script, args, timeout=timeout):
            RESULTS[label] = RESULTS[tag]
            return True
        result = RESULTS[tag]
        if result['returncode'] != -9 or result['timed_out']:
            reason = 'timed out' if result['timed_out'] else f'rc={result["returncode"]}'
            print(f'[{label}] {reason} is not a resumable OOM; not retrying', flush=True)
            RESULTS[label] = result
            return False
    print(f'[{label}] still failing after {attempts} attempts', flush=True)
    RESULTS[label] = RESULTS[tag]
    return False

def free_vram():
    import gc
    gc.collect()
    torch.cuda.empty_cache()
    # GPU allocation is released between schemes; host memory ratchets upward
    # across model loads and is what eventually triggers Colab's OOM killer.
    host = ''
    try:
        for line in pathlib.Path('/proc/meminfo').read_text().splitlines():
            if line.startswith('MemAvailable:'):
                host = f', host available {float(line.split()[1]) / 1e6:.1f} GB'
    except OSError:
        pass
    print(f'[vram] {torch.cuda.memory_allocated()/1e9:.2f} GB allocated{host}')

def checkpoint_to_drive():
    '''Mirror completed run directories to Drive; caches already live there.'''
    if not DRIVE_ROOT.exists():
        return
    import shutil
    for name in ('runs', 'behavioural_cache', 'sector_cache'):
        src = pathlib.Path('artifacts') / name
        if src.exists():
            shutil.copytree(src, DRIVE_ROOT / 'artifacts' / name, dirs_exist_ok=True)
    print(f'[drive] mirrored artifacts/ to {DRIVE_ROOT}')

def restore_from_drive():
    '''Bring back prior run directories before anything runs.'''
    if not DRIVE_ROOT.exists():
        return
    import shutil
    src = DRIVE_ROOT / 'artifacts' / 'runs'
    if src.exists():
        shutil.copytree(src, pathlib.Path('artifacts') / 'runs', dirs_exist_ok=True)
        print('[drive] restored artifacts/runs')

restore_from_drive()

def latest_run(pattern):
    hits = sorted(pathlib.Path('artifacts/runs').glob(pattern))
    return hits[-1] if hits else None

def completed_run(label, schemes):
    hits, complete = sorted(pathlib.Path('artifacts/runs').glob(f'*_{label}')), []
    for run in hits:
        try:
            manifest = json.loads((run / 'manifest.json').read_text(encoding='utf-8'))
        except (OSError, json.JSONDecodeError):
            continue
        if list(manifest.get('schemes', [])) == schemes and all((run / 'results' / f'completions_{s}.json').exists() for s in schemes):
            complete.append(run)
    if len(complete) > 1:
        raise SystemExit(f'multiple completed runs share {label}: {[p.name for p in complete]}. Refusing to choose one.')
    return complete[0] if complete else None

def print_pairing(run):
    manifest = json.loads((run / 'manifest.json').read_text(encoding='utf-8'))
    digest = manifest.get('corpora', {}).get('prompts', {}).get('sha256_ordered')
    print(f'[run] {run}')
    print(f'[pairing] corpora.prompts.sha256_ordered = {digest}')

def record_skip(label, reason):
    RESULTS[label] = {'returncode': 0, 'skipped': True, 'minutes': 0.0, 'tail': [reason]}
    print(f'[{label}] already complete after Drive restore; skipping: {reason}')

def grade_complete(run, filename):
    return (run / 'results' / filename).exists()


## Step 1 — long-window HH-RLHF

This answers the headline question: at 256 generated tokens, does the FP16-versus-4-bit behavioural comparison hold for Qwen2.5-3B and Phi-3.5-mini?

In [ ]:
for tag, model in MODELS_LONG:
    run_label, step_label = f'r3-long256-{tag}', f'long-{tag}'
    run_dir = completed_run(run_label, ['FP16', 'RTN_4B'])
    if run_dir is None:
        print(f'\n{step_label}: will produce 500 HH-RLHF completions at 256 tokens for FP16 and RTN 4-bit; it settles the long-window headline for {model}.')
        free_vram()
        run_step_resumable(step_label, 'run_behavioural_ladder.py', ['--model', model, '--n', str(N_LONG), '--bits', '4', '--max-new-tokens', str(LONG_TOKENS), '--seed', str(SEED), '--no-activations', '--cache', BEHAV_CACHE, '--label', run_label], timeout=75 * 60)
        checkpoint_to_drive()
        run_dir = completed_run(run_label, ['FP16', 'RTN_4B'])
    else:
        record_skip(step_label, str(run_dir))
    if run_dir is None:
        RESULTS[f'judge-long-{tag}'] = {'returncode': -1, 'blocked': True, 'minutes': 0.0, 'tail': ['generation failed']}
        print(f'[judge-long-{tag}] not run because its generation did not finish')
        checkpoint_to_drive()
        continue
    print_pairing(run_dir)
    judge_label = f'judge-long-{tag}'
    if grade_complete(run_dir, 'judge_classification.json'):
        record_skip(judge_label, str(run_dir / 'results' / 'judge_classification.json'))
    else:
        print(f'{judge_label}: will produce three-way 7B-NF4 judgements over saved 256-token completions; it settles whether the headline depends on the phrase marker.')
        free_vram()
        # This grader has no --max-length argparse argument; passing one would make an unattended run fail.
        run_step_resumable(judge_label, 'classify_completions_judge.py', [str(run_dir), '--judge-model', JUDGE_MODEL, '--judge-4bit', '--completion-chars', str(JUDGE_COMPLETION_CHARS), '--batch-size', '4'], timeout=30 * 60)
        checkpoint_to_drive()


## Step 2 — long-window XSTest baselines

This closes the labelled-arm gap: at 256 generated tokens, what are the FP16 baselines for harmful and benign XSTest prompts across the three models?

In [ ]:
for tag, model in MODELS_XSTEST:
    run_label, step_label = f'r3-xstest256-{tag}', f'xstest-{tag}'
    run_dir = completed_run(run_label, ['FP16'])
    if run_dir is None:
        print(f'\n{step_label}: will produce 300 labelled XSTest completions at 256 tokens for FP16; it supplies the labelled long-window baseline for {model}.')
        free_vram()
        run_step_resumable(step_label, 'run_behavioural_ladder.py', ['--model', model, '--prompts', 'data/eval_suites/xstest.jsonl', '--n', str(N_XSTEST), '--bits', '--max-new-tokens', str(LONG_TOKENS), '--seed', str(SEED), '--no-activations', '--cache', BEHAV_CACHE, '--label', run_label], timeout=25 * 60)
        checkpoint_to_drive()
        run_dir = completed_run(run_label, ['FP16'])
    else:
        record_skip(step_label, str(run_dir))
    if run_dir is None:
        RESULTS[f'taxonomy-xstest-{tag}'] = {'returncode': -1, 'blocked': True, 'minutes': 0.0, 'tail': ['generation failed']}
        print(f'[taxonomy-xstest-{tag}] not run because its generation did not finish')
        checkpoint_to_drive()
        continue
    print_pairing(run_dir)
    grade_label = f'taxonomy-xstest-{tag}'
    if grade_complete(run_dir, 'completion_taxonomy.json'):
        record_skip(grade_label, str(run_dir / 'results' / 'completion_taxonomy.json'))
    else:
        print(f'{grade_label}: will produce five-way 7B-NF4 taxonomy judgements over labelled 256-token completions; it separates refusal, deflection, disclaimer, compliance and unclear responses.')
        free_vram()
        run_step_resumable(grade_label, 'classify_completion_taxonomy.py', [str(run_dir), '--judge-model', JUDGE_MODEL, '--judge-4bit', '--completion-chars', str(JUDGE_COMPLETION_CHARS), '--max-length', str(TAXONOMY_MAX_LENGTH), '--batch-size', '4'], timeout=30 * 60)
        checkpoint_to_drive()


## Export

The archive contains only round-three run directories, with paths relative to the repository root. A failed or blocked step makes the archive name begin with INCOMPLETE and prints the scientific question that remains unanswered.

In [ ]:
import zipfile
runs = sorted(pathlib.Path('artifacts/runs').glob('*_r3-*'))
print('round-three run directories:')
for run in runs:
    print('  ', run.name)

steps = {key: value for key, value in RESULTS.items() if '-retry' not in key}
failed = [key for key, value in steps.items() if value.get('returncode') != 0]
retried = sorted({key.split('-retry')[0] for key in RESULTS if '-retry' in key})
status = {'steps': RESULTS, 'runs': [run.name for run in runs], 'failed': failed, 'retried': retried, 'long_models': [tag for tag, _ in MODELS_LONG], 'xstest_models': [tag for tag, _ in MODELS_XSTEST], 'n_long_per_class': N_LONG, 'n_xstest_per_class': N_XSTEST, 'max_new_tokens': LONG_TOKENS, 'completion_chars': JUDGE_COMPLETION_CHARS, 'taxonomy_max_length': TAXONOMY_MAX_LENGTH}
status_path = pathlib.Path('artifacts/runs/ROUND3_STATUS.json')
status_path.write_text(json.dumps(status, indent=2), encoding='utf-8')
checkpoint_to_drive()

stamp = time.strftime('%Y%m%d-%H%M%S')
prefix = 'INCOMPLETE_' if failed else ''
archive = pathlib.Path((f'/content/{prefix}cliffguard_round3_{stamp}.zip' if IN_COLAB else f'{prefix}cliffguard_round3_{stamp}.zip'))
# The status JSON goes IN the archive, not just beside it. The archive is what
# leaves this machine; a reader who has only the zip must be able to tell which
# steps ran, which failed, and which were restored from a previous session --
# otherwise a step that never ran is indistinguishable from one that ran and
# found nothing, which is the single most dangerous confusion this project has.
with zipfile.ZipFile(archive, 'w', zipfile.ZIP_DEFLATED) as zf:
    zf.write(status_path, status_path.as_posix())
    for run in runs:
        for path in sorted(run.rglob('*')):
            if path.is_file():
                zf.write(path, path.as_posix())
print(f'\nwrote {archive}  ({archive.stat().st_size / 1e6:.1f} MB)')
print(f'wrote status JSON: {status_path}')
print('\nSTEPS:')
for key, value in steps.items():
    mark = 'ok' if value.get('returncode') == 0 else f'FAIL({value.get("returncode")})'
    suffix = ' (restored)' if value.get('skipped') else ''
    print(f'  {mark:9s} {key:28s} {value.get("minutes", 0):6.1f} min{suffix}')
if failed:
    print('\nINCOMPLETE RUN — do not report a missing comparison as a null result.')
    if any(key.startswith('long-') or key.startswith('judge-long-') for key in failed):
        print('UNANSWERED: whether the 256-token HH-RLHF FP16-versus-4-bit headline holds for every requested model.')
    if any(key.startswith('xstest-') or key.startswith('taxonomy-xstest-') for key in failed):
        print('UNANSWERED: the labelled 256-token XSTest FP16 baseline and its refusal/deflection taxonomy for every requested model.')
else:
    print('\nAll requested round-three measurements and grades completed.')
if retried:
    print('resumed after an OOM:', retried)
if IN_COLAB:
    try:
        from google.colab import files
        files.download(str(archive))
    except Exception as exc:
        print('download it from the file browser instead:', exc)
